<a href="https://colab.research.google.com/github/FurkanGozukara/Stable-Diffusion/blob/main/ColabNotebooks/1_click_deep_fake_for_free_by_SECourses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Updated 1 September 2026 - tested on the current Colab GPU runtime
### Fresh setup verified with Python 3.13, Torch 2.11, CUDA 12.8, and a T4 GPU
### Someone upgraded to Gold Tier and I fixed all issues : https://www.patreon.com/c/SECourses
## Most Advanced DeepFake FaceFusion for Windows, RunPod and Massed Compute : https://www.patreon.com/posts/103765029
## Very Advanced VisoMaster for Windows and Massed Compute : https://www.patreon.com/posts/121570322
## Deep Live Cam for Windows : https://www.patreon.com/posts/125826778

## If notebook gets broken get a membership on Patreon and message me for fixing

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import subprocess
import sys

ROOP_DIR = Path('/content/roop')
BASICSR_DIR = ROOP_DIR / 'BasicSR'

def run_live(command, cwd=None):
    command = [str(part) for part in command]
    print('\n$ ' + ' '.join(command), flush=True)
    with subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=0,
    ) as process:
        pending = []
        assert process.stdout is not None
        while True:
            character = process.stdout.read(1)
            if character == '':
                break
            pending.append(character)
            if character in '\r\n' or len(pending) >= 256:
                sys.stdout.write(''.join(pending))
                sys.stdout.flush()
                pending.clear()
        if pending:
            sys.stdout.write(''.join(pending))
            sys.stdout.flush()
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def show_download_button(path):
    from google.colab import files
    from IPython.display import display
    import ipywidgets as widgets

    output_file = Path(path)
    button = widgets.Button(
        description='Download output', icon='download', button_style='success',
        layout=widgets.Layout(width='190px'),
    )
    status = widgets.Output()

    def download(_):
        with status:
            status.clear_output(wait=True)
            if not output_file.is_file():
                print(f'Generated output not found: {output_file}')
                return
            print(f'Preparing download: {output_file.name}')
            files.download(str(output_file))

    button.on_click(download)
    display(widgets.VBox([
        widgets.HTML(f'<code>{output_file}</code>'), button, status,
    ]))

def clone_or_update(url, destination, branch):
    destination = Path(destination)
    if (destination / '.git').is_dir():
        print(f'Updating {destination.name}...')
        run_live(['git', '-C', str(destination), 'pull', '--ff-only', '--progress'])
        return
    if destination.exists():
        raise RuntimeError(f'{destination} exists but is not a Git checkout. Rename or remove it, then rerun this cell.')
    run_live(['git', 'clone', '--progress', '--depth', '1', '--branch', branch, url, str(destination)])

print('Step 1/4: cloning or updating repositories...', flush=True)
clone_or_update('https://github.com/FurkanGozukara/rop_fixed.git', ROOP_DIR, 'main')
clone_or_update('https://github.com/FurkanGozukara/BasicSR.git', BASICSR_DIR, 'master')

# Install the patched BasicSR checkout as a regular package so it is immediately importable
# in this kernel and satisfies GFPGAN without fetching the incompatible PyPI source.
print('Step 2/4: installing the patched BasicSR package...', flush=True)
run_live([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'on', '--no-deps', '--force-reinstall', str(BASICSR_DIR)])
print('Step 3/4: installing runtime dependencies...', flush=True)
run_live([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'on', '-r', str(ROOP_DIR / 'a.txt')])
print('Applying Colab headless OpenCV compatibility patch...', flush=True)
import importlib.util
opennsfw2_spec = importlib.util.find_spec('opennsfw2')
if opennsfw2_spec is None or opennsfw2_spec.origin is None:
    raise RuntimeError('opennsfw2 was not installed correctly.')
inference_file = Path(opennsfw2_spec.origin).parent / '_inference.py'
inference_source = inference_file.read_text(encoding='utf-8')
headless_call = 'cv2.destroyAllWindows()'
if headless_call in inference_source:
    inference_file.write_text(
        inference_source.replace(headless_call, 'None  # Colab uses headless OpenCV'),
        encoding='utf-8',
    )
    print(f'Patched GUI-only OpenCV cleanup in {inference_file}')
else:
    print('Headless OpenCV compatibility patch is already applied.')


print('Step 4/4: validating the GPU runtime...', flush=True)
import basicsr
import onnxruntime as ort

providers = ort.get_available_providers()
if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError(f'CUDAExecutionProvider is unavailable. Select a GPU runtime and rerun this cell. Providers: {providers}')

print('Setup complete.')
print('Python:', sys.version.split()[0])
print('ONNX Runtime:', ort.__version__)
print('Providers:', providers)
print('TensorFlow:', metadata.version('tensorflow'))
print('BasicSR:', basicsr.__version__)

**Use either method: enter existing source/target paths in a processing cell, or use the optional upload buttons directly below. Uploaded files automatically override the two manual input paths; clear an uploaded-path box to use the manual path again.**

**Setup logs, model downloads, frame processing, and FFmpeg extraction/encoding progress are streamed live. The first processing run also downloads the InsightFace and GFPGAN model weights.**

**Quality 1 gives the best quality and largest file; quality 100 gives the lowest quality and smallest file. Temporary frames are removed by default to save disk space.**

In [ ]:
# Optional uploads. Run this cell, then use either button below.
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

UPLOAD_DIR = Path('/content/roop/uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

source_upload = widgets.FileUpload(
    accept='image/*', multiple=False, description='Upload source image',
    button_style='primary', layout=widgets.Layout(width='220px'),
)
target_upload = widgets.FileUpload(
    accept='image/*,video/*', multiple=False, description='Upload target media',
    button_style='primary', layout=widgets.Layout(width='220px'),
)
source_upload_path = widgets.Text(
    description='Source:', placeholder='No upload selected; manual path will be used',
    layout=widgets.Layout(width='70%'), style={'description_width': '60px'},
)
target_upload_path = widgets.Text(
    description='Target:', placeholder='No upload selected; manual path will be used',
    layout=widgets.Layout(width='70%'), style={'description_width': '60px'},
)
upload_status = widgets.Output()

def _latest_upload(value):
    if not value:
        return None
    return list(value.values())[-1] if isinstance(value, dict) else value[-1]

def _save_upload(change, uploader, path_widget, label):
    entry = _latest_upload(change['new'])
    if entry is None:
        return
    if hasattr(entry, 'get'):
        name = entry.get('name') or entry.get('metadata', {}).get('name')
        content = entry.get('content')
    else:
        name = getattr(entry, 'name', None)
        content = getattr(entry, 'content', None)
    if not name or content is None:
        raise RuntimeError(f'Could not read the selected {label} file.')
    destination = UPLOAD_DIR / Path(name).name
    destination.write_bytes(bytes(content))
    path_widget.value = str(destination)
    with upload_status:
        upload_status.clear_output(wait=True)
        print(f'{label.title()} ready: {destination} ({destination.stat().st_size / (1024 * 1024):.1f} MiB)')
    try:
        uploader.value = {} if isinstance(uploader.value, dict) else ()
    except Exception:
        pass

source_upload.observe(
    lambda change: _save_upload(change, source_upload, source_upload_path, 'source'), names='value'
)
target_upload.observe(
    lambda change: _save_upload(change, target_upload, target_upload_path, 'target'), names='value'
)

display(widgets.VBox([
    widgets.HTML('<b>Optional uploads</b> - the saved path overrides the matching manual path below.'),
    widgets.HBox([source_upload, source_upload_path]),
    widgets.HBox([target_upload, target_upload_path]),
    upload_status,
]))

In [ ]:
from pathlib import Path
import sys

source_path = '/content/roop/face2.png' # @param {"type":"string"}
target_path = '/content/roop/test_video.mp4' # @param {"type":"string"}
output_path = '/content/roop/face_changed_video_v2.mp4' # @param {"type":"string"}
keep_frames = False # @param {"type":"boolean"}

if 'run_live' not in globals():
    raise RuntimeError('Run the setup cell first to install dependencies and enable live progress.')
uploaded_source = globals().get('source_upload_path')
uploaded_target = globals().get('target_upload_path')
if uploaded_source is not None and uploaded_source.value.strip():
    source_path = uploaded_source.value.strip()
    print(f'Using uploaded source: {source_path}', flush=True)
if uploaded_target is not None and uploaded_target.value.strip():
    target_path = uploaded_target.value.strip()
    print(f'Using uploaded target: {target_path}', flush=True)

for label, path in [('source', source_path), ('target', target_path)]:
    if not Path(path).is_file():
        raise FileNotFoundError(f'The {label} file does not exist: {path}')

command = [
    sys.executable, '-u', 'run.py',
    '-s', source_path, '-t', target_path, '-o', output_path,
    '--keep-fps', '--temp-frame-quality', '1', '--output-video-quality', '1',
    '--execution-provider', 'cuda',
]
if keep_frames:
    command.append('--keep-frames')
run_live(command, cwd='/content/roop')
if not Path(output_path).is_file():
    raise RuntimeError(f'Processing finished without creating {output_path}')
print(f'Finished: {output_path}')
show_download_button(output_path)


$ /usr/bin/python3 -u run.py -s /content/roop/face2.png -t /content/roop/test_video.mp4 -o /content/roop/face_changed_video_v2.mp4 --keep-fps --temp-frame-quality 1 --output-video-quality 1 --execution-provider cuda
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'c

**Below code will do also face restoration to improve quality significantly but it will take longer**

In [ ]:
from pathlib import Path
import sys

source_path = '/content/roop/face2.png' # @param {"type":"string"}
target_path = '/content/roop/test_video.mp4' # @param {"type":"string"}
output_path = '/content/roop/face_restored_video3.mp4' # @param {"type":"string"}
keep_frames = False # @param {"type":"boolean"}

if 'run_live' not in globals():
    raise RuntimeError('Run the setup cell first to install dependencies and enable live progress.')
uploaded_source = globals().get('source_upload_path')
uploaded_target = globals().get('target_upload_path')
if uploaded_source is not None and uploaded_source.value.strip():
    source_path = uploaded_source.value.strip()
    print(f'Using uploaded source: {source_path}', flush=True)
if uploaded_target is not None and uploaded_target.value.strip():
    target_path = uploaded_target.value.strip()
    print(f'Using uploaded target: {target_path}', flush=True)

for label, path in [('source', source_path), ('target', target_path)]:
    if not Path(path).is_file():
        raise FileNotFoundError(f'The {label} file does not exist: {path}')

command = [
    sys.executable, '-u', 'run.py',
    '-s', source_path, '-t', target_path, '-o', output_path,
    '--keep-fps', '--temp-frame-quality', '1', '--output-video-quality', '1',
    '--execution-provider', 'cuda',
    '--frame-processor', 'face_swapper', 'face_enhancer',
]
if keep_frames:
    command.append('--keep-frames')
run_live(command, cwd='/content/roop')
if not Path(output_path).is_file():
    raise RuntimeError(f'Processing finished without creating {output_path}')
print(f'Finished: {output_path}')
show_download_button(output_path)

### All options are displayed below
Append any of them to the above commands before executing
```
python run.py [options]

-h, --help                                                                 show this help message and exit
-s SOURCE_PATH, --source SOURCE_PATH                                       select an source image
-t TARGET_PATH, --target TARGET_PATH                                       select an target image or video
-o OUTPUT_PATH, --output OUTPUT_PATH                                       select output file or directory
--frame-processor FRAME_PROCESSOR [FRAME_PROCESSOR ...]                    frame processors (choices: face_swapper, face_enhancer, ...)
--keep-fps                                                                 keep target fps
--keep-frames                                                              keep temporary frames
--skip-audio                                                               skip target audio
--many-faces                                                               process every face
--reference-face-position REFERENCE_FACE_POSITION                          position of the reference face
--reference-frame-number REFERENCE_FRAME_NUMBER                            number of the reference frame
--similar-face-distance SIMILAR_FACE_DISTANCE                              face distance used for recognition
--temp-frame-format {jpg,png}                                              image format used for frame extraction
--temp-frame-quality [0-100]                                               image quality used for frame extraction
--output-video-encoder {libx264,libx265,libvpx-vp9,h264_nvenc,hevc_nvenc}  encoder used for the output video
--output-video-quality [0-100]                                             quality used for the output video
--max-memory MAX_MEMORY                                                    maximum amount of RAM in GB
--execution-provider {tensorrt,cuda,cpu} [{tensorrt,cuda,cpu} ...]          available execution provider
--execution-threads EXECUTION_THREADS                                      number of execution threads
-v, --version                                                              show program's version number and exit
  ```

### Download a generated output

In [ ]:
output_path = '/content/roop/face_changed_video_v2.mp4' # @param {"type":"string"}
if 'show_download_button' not in globals():
    raise RuntimeError('Run the setup cell first to enable the download button.')
show_download_button(output_path)


In [ ]:
# Optional public smoke-test inputs. Run this before a processing cell.
from pathlib import Path
from urllib.request import urlretrieve

examples = {
    'face2.png': 'https://huggingface.co/MonsterMMORPG/examples2/resolve/main/face2.png',
    'test_video.mp4': 'https://huggingface.co/MonsterMMORPG/examples2/resolve/main/test_video.mp4',
}
for filename, url in examples.items():
    destination = Path('/content/roop') / filename
    if destination.is_file():
        print(f'Reusing {destination}')
    else:
        print(f'Downloading {filename}...')
        urlretrieve(url, destination)
print('Example inputs are ready.')